# argentina.fechas — Pruebas interactivas

Recorrido paso a paso del módulo `argentina.fechas`.

Funciones para parsear formatos argentinos (`dd/mm/yyyy`, `dd-mm-yyyy`, `yyyy-mm-dd`, etc.), normalizar a ISO, calcular edad y armar agrupaciones típicas (cohorte, año lectivo, mes-año). Solo stdlib — sin pandas, sin feriados, sin calendarios oficiales.

## 1. Setup

In [1]:
import argentina as arg
from datetime import date, datetime

print(f"argentina v{arg.__version__}")
print(f"formatos soportados: {arg.fechas.FORMATOS_FECHA}")

argentina v0.0.17
formatos soportados: ('%d/%m/%Y', '%d-%m-%Y', '%Y-%m-%d', '%Y/%m/%d', '%d/%m/%y', '%d-%m-%y')


## 2. parsear_fecha

Acepta los formatos más frecuentes en bases argentinas y devuelve un `date`.

In [2]:
for v in ["31/12/2024", "31-12-2024", "2024-12-31", "2024/12/31", "31/12/24", "31-12-24"]:
    print(f"{v!r:14} → {arg.fechas.parsear_fecha(v)!r}")

'31/12/2024'   → datetime.date(2024, 12, 31)
'31-12-2024'   → datetime.date(2024, 12, 31)
'2024-12-31'   → datetime.date(2024, 12, 31)
'2024/12/31'   → datetime.date(2024, 12, 31)
'31/12/24'     → datetime.date(2024, 12, 31)
'31-12-24'     → datetime.date(2024, 12, 31)


In [3]:
# date / datetime pasan transparente
print(arg.fechas.parsear_fecha(date(2024, 12, 31)))
print(arg.fechas.parsear_fecha(datetime(2024, 12, 31, 14, 30)))

2024-12-31
2024-12-31


In [4]:
# Inválidos → None
print(arg.fechas.parsear_fecha("31/31/2024"))    # mes inválido
print(arg.fechas.parsear_fecha("mala"))
print(arg.fechas.parsear_fecha(""))
print(arg.fechas.parsear_fecha(None))

None
None
None
None


## 3. es_fecha_valida / fecha_iso

Atajos sobre `parsear_fecha`: bool y string ISO.

In [5]:
for v in ["31/12/2024", "31/02/2024", "mala", None]:
    print(f"{v!r:14} valida={arg.fechas.es_fecha_valida(v)}  iso={arg.fechas.fecha_iso(v)!r}")

'31/12/2024'   valida=True  iso='2024-12-31'
'31/02/2024'   valida=False  iso=None
'mala'         valida=False  iso=None
None           valida=False  iso=None


## 4. edad_en_anios

Calcula años completos. Respeta si ya cumplió en la fecha de referencia.

In [6]:
# Caso típico: ya cumplió
arg.fechas.edad_en_anios("10/05/2015", "12/05/2026")

11

In [7]:
# Antes del cumple del año actual → resta 1
arg.fechas.edad_en_anios("10/12/2015", "12/05/2026")

10

In [8]:
# Si no le pasás referencia, usa hoy
arg.fechas.edad_en_anios("01/01/2000")

26

In [9]:
# Casos borde → None
print(arg.fechas.edad_en_anios("12/05/2030", "12/05/2026"))   # nacimiento futuro
print(arg.fechas.edad_en_anios("mala", "12/05/2026"))
print(arg.fechas.edad_en_anios(None, "12/05/2026"))

None
None
None


## 5. cohorte_nacimiento

Atajo: te devuelve solo el año. Útil para agrupar nacimientos por cohorte sin mezclar mes/día.

In [10]:
for v in ["10/05/2015", "01/01/1985", "31/12/2000", "mala", None]:
    print(f"{v!r:14} → {arg.fechas.cohorte_nacimiento(v)}")

'10/05/2015'   → 2015
'01/01/1985'   → 1985
'31/12/2000'   → 2000
'mala'         → None
None           → None


## 6. anio_lectivo

El ciclo escolar argentino arranca en **marzo**. Enero y febrero son receso, así que se asignan al ciclo del año anterior. El mes de inicio es configurable.

In [11]:
# Default: arranca en marzo
for v in ["15/01/2024", "15/02/2024", "01/03/2024", "15/03/2024", "30/12/2024"]:
    print(f"{v!r:14} → ciclo lectivo {arg.fechas.anio_lectivo(v)}")

'15/01/2024'   → ciclo lectivo 2023
'15/02/2024'   → ciclo lectivo 2023
'01/03/2024'   → ciclo lectivo 2024
'15/03/2024'   → ciclo lectivo 2024
'30/12/2024'   → ciclo lectivo 2024


In [12]:
# Configurable: si una jurisdicción arranca en febrero, pasalo
for v in ["15/01/2024", "15/02/2024", "15/03/2024"]:
    print(f"{v!r} default={arg.fechas.anio_lectivo(v)} feb-start={arg.fechas.anio_lectivo(v, mes_inicio=2)}")

'15/01/2024' default=2023 feb-start=2023
'15/02/2024' default=2023 feb-start=2024
'15/03/2024' default=2024 feb-start=2024


## 7. mes_anio

Reformatea a `YYYY-MM` para usar como clave de agrupación mensual (típico en series administrativas).

In [13]:
for v in ["31/12/2024", "01/03/2024", "15/07/2020", "2024-01-05", "mala", None]:
    print(f"{v!r:14} → {arg.fechas.mes_anio(v)!r}")

'31/12/2024'   → '2024-12'
'01/03/2024'   → '2024-03'
'15/07/2020'   → '2020-07'
'2024-01-05'   → '2024-01'
'mala'         → None
None           → None


## 8. Combinando todo

Pipeline típico: filas crudas con fecha de nacimiento y fecha de evento, normalizar y agregar campos derivados.

In [14]:
registros = [
    {"nacimiento": "10/05/2015", "evento": "15/03/2024"},
    {"nacimiento": "22/11/2010", "evento": "15/02/2024"},
    {"nacimiento": "01/01/2000", "evento": "31/12/2024"},
    {"nacimiento": "mala",       "evento": "15/03/2024"},
]

for r in registros:
    print({
        "nac_iso":      arg.fechas.fecha_iso(r["nacimiento"]),
        "cohorte":      arg.fechas.cohorte_nacimiento(r["nacimiento"]),
        "edad":         arg.fechas.edad_en_anios(r["nacimiento"], r["evento"]),
        "anio_lectivo": arg.fechas.anio_lectivo(r["evento"]),
        "mes_evento":   arg.fechas.mes_anio(r["evento"]),
    })

{'nac_iso': '2015-05-10', 'cohorte': 2015, 'edad': 8, 'anio_lectivo': 2024, 'mes_evento': '2024-03'}
{'nac_iso': '2010-11-22', 'cohorte': 2010, 'edad': 13, 'anio_lectivo': 2023, 'mes_evento': '2024-02'}
{'nac_iso': '2000-01-01', 'cohorte': 2000, 'edad': 24, 'anio_lectivo': 2024, 'mes_evento': '2024-12'}
{'nac_iso': None, 'cohorte': None, 'edad': None, 'anio_lectivo': 2024, 'mes_evento': '2024-03'}


## 9. Tests automáticos

```bash
cd /Users/tobiasyatche/argentina
pytest tests/test_fechas.py -v
```

## Notas sueltas / TODOs

- Solo stdlib (`datetime`). Sin pandas, sin `dateutil`, sin feriados, sin calendarios oficiales.
- `parsear_fecha` recorre los formatos en orden y se queda con el primero que matchea — el orden de `FORMATOS_FECHA` es deliberado para favorecer el formato argentino (`dd/mm/yyyy`) sobre el ISO cuando son ambiguos.
- El año de 2 dígitos (`%y`) usa el cutoff de Python (00–68 → 2000s, 69–99 → 1900s). Para datos con DNI o nacimientos, eso suele ser razonable; ojo si tu base mezcla.
- `anio_lectivo` no contempla cierre lectivo en diciembre — todo lo que cae después de marzo se asigna al año en curso. Si en algún momento se quiere distinguir entre "ciclo en curso" y "ciclo cerrado", agregar un parámetro `mes_cierre`.
- Feriados, no laborables y calendarios oficiales (escolar provincial, fiscal, judicial) quedan explícitamente fuera del scope por ahora.